# V4: V3 pipeline + fine-tuned `labeltune-8b` (corrected)
Same retrieval pipeline as V3. Changes vs the broken V4 run:
- Installs everything at the top (packages, Ollama, models). Registers the tuned model straight from the merged folder on Drive.
- Prompt matches the training format exactly (system prompt in `system`, user = `EXCERPTS / QUESTION / ANSWER`). No few-shot (optional flag).
- Parser reads the tuned model's `Reasoning: ... \n\n- bullet [S1]` output and keeps the reasoning out of the claims.
- Stop sequences added, output budget raised, truncated answers retried once.
- Query-embedding shape fixed (flat vector).
- Split and verify steps use base `llama3:8b`, so only the answer step changes versus V3.

In [2]:
# ===== 0. packages, Drive, Ollama, models =====
import os, sys, subprocess, time, shutil, requests
os.environ["JAX_PLATFORMS"] = "cpu"; os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"

def sh(cmd, show=True, cwd=None):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, cwd=cwd)
    if show: print((r.stdout or "")[-1000:], (r.stderr or "")[-1000:])
    return r

print("pip rc:", sh("pip install -q faiss-cpu bm25s networkx pandas numpy tqdm requests scikit-learn "
                    "sentence-transformers sentencepiece protobuf gguf", show=False).returncode)

from google.colab import drive
drive.mount("/content/drive")
assert os.path.isdir("/content/drive/MyDrive"), "Drive is not mounted"

if not shutil.which("ollama"):                     # fresh Colab runtime has no Ollama
    sh("apt-get update -qq && apt-get install -y -qq zstd", show=False)
    sh("curl -fsSL https://ollama.com/install.sh | sh")
assert shutil.which("ollama"), "Ollama install failed, read the output above"

def ollama_up():
    try: return requests.get("http://localhost:11434/api/tags", timeout=3).status_code == 200
    except requests.RequestException: return False

if not ollama_up():
    env = dict(os.environ, OLLAMA_MAX_LOADED_MODELS="3", OLLAMA_NUM_PARALLEL="1", OLLAMA_KEEP_ALIVE="30m")
    subprocess.Popen(["ollama", "serve"], stdout=open("/content/ollama.log", "a"), stderr=subprocess.STDOUT,
                     env=env, start_new_session=True)
    for _ in range(90):
        if ollama_up(): break
        time.sleep(1)
assert ollama_up(), "Ollama did not start:\n" + open("/content/ollama.log").read()[-1500:]

for m in ["nomic-embed-text", "llama3:8b"]:        # embeddings + base model for split/verify
    print(m, "pull rc =", sh(f"ollama pull {m}", show=False).returncode)
print(sh("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader", show=False).stdout or "NO GPU: this will be very slow")

pip rc: 0
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
          89.6%
#################################################################         91.6%
##################################################################        92.3%
###################################################################       93.8%
####################################################################      95.3%
#####################################################################     96.7%
######################################################################    97.7%
######################################################################    98.3%
#######################################################################   99.6%
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating 

In [3]:
# ===== 1. register the fine-tuned model from the merged folder on Drive =====
MERGED_DIR = "/content/drive/MyDrive/GraphRAG_FineTune/merged_labeltune_8b"
MODEL_NAME = "labeltune-8b"
assert os.path.isfile(f"{MERGED_DIR}/config.json"), f"merged model not found: {MERGED_DIR}"
assert "quantization_config" not in open(f"{MERGED_DIR}/config.json").read(), "merged model is 4-bit; re-merge in bf16"

MODELFILE_TXT = """FROM <<FROM>>
TEMPLATE """ + '"""' + """{{ if .System }}<|start_header_id|>system<|end_header_id|>

{{ .System }}<|eot_id|>{{ end }}{{ if .Prompt }}<|start_header_id|>user<|end_header_id|>

{{ .Prompt }}<|eot_id|>{{ end }}<|start_header_id|>assistant<|end_header_id|>

{{ .Response }}<|eot_id|>""" + '"""' + """
PARAMETER temperature 0.1
PARAMETER num_ctx 8192
PARAMETER stop "<|eot_id|>"
"""

def registered(): return MODEL_NAME in sh("ollama list", show=False).stdout

def create(frm, extra=""):
    open("/content/Modelfile", "w").write(MODELFILE_TXT.replace("<<FROM>>", frm))
    return sh(f"ollama create {MODEL_NAME} -f /content/Modelfile {extra}", cwd="/content")

if not registered():
    # route 1: Ollama imports the safetensors folder and quantises to 4-bit itself (no llama.cpp needed)
    create(MERGED_DIR, "--quantize q4_K_M")
if not registered():
    # route 2: convert with llama.cpp to an 8-bit GGUF on local disk, then import
    print("route 1 failed, trying llama.cpp conversion (10-20 min) ...")
    if not os.path.exists("/content/llama.cpp"):
        sh("git clone --depth 1 https://github.com/ggerganov/llama.cpp /content/llama.cpp", show=False)
    gg = "/content/labeltune-8b-q8_0.gguf"
    sh(f"python /content/llama.cpp/convert_hf_to_gguf.py {MERGED_DIR} --outfile {gg} --outtype q8_0")
    assert os.path.exists(gg), "GGUF conversion failed, read the error above"
    create(gg)
assert registered(), "labeltune-8b not registered, read the errors above"
print(sh("ollama list", show=False).stdout)

 
Error: unsupported --quantize "q4_K_M": supported types are int4, int8, nvfp4, mxfp4, mxfp8

route 1 failed, trying llama.cpp conversion (10-20 min) ...
 .0Mbyte/s]
Writing:  95%|█████████▍| 8.06G/8.53G [05:30<00:23, 19.5Mbyte/s]
Writing:  95%|█████████▍| 8.07G/8.53G [05:31<00:28, 16.1Mbyte/s]
Writing:  95%|█████████▌| 8.13G/8.53G [05:33<00:18, 22.0Mbyte/s]
Writing:  96%|█████████▌| 8.19G/8.53G [05:35<00:13, 25.6Mbyte/s]
Writing:  97%|█████████▋| 8.26G/8.53G [05:38<00:11, 24.9Mbyte/s]
Writing:  97%|█████████▋| 8.28G/8.53G [05:38<00:09, 28.1Mbyte/s]
Writing:  97%|█████████▋| 8.30G/8.53G [05:38<00:07, 31.5Mbyte/s]
Writing:  98%|█████████▊| 8.36G/8.53G [05:40<00:05, 33.6Mbyte/s]
Writing:  99%|█████████▊| 8.43G/8.53G [05:42<00:03, 32.5Mbyte/s]
Writing:  99%|█████████▉| 8.49G/8.53G [05:44<00:01, 32.1Mbyte/s]
Writing: 100%|█████████▉| 8.49G/8.53G [05:46<00:01, 24.0Mbyte/s]
Writing: 100%|█████████▉| 8.51G/8.53G [05:46<00:00, 27.3Mbyte/s]
Writing: 100%|█████████▉| 8.53G/8.53G [05:47<00:00, 2

In [4]:
# ===== 2. config (V3 settings; changes are marked CHANGED) =====
import re, json, pickle, glob, hashlib, html, traceback
from collections import defaultdict, Counter
import numpy as np, pandas as pd, faiss, bm25s
from tqdm.auto import tqdm

BASE_DIR = "/content/drive/MyDrive/Research_kgTxagent/graph_rag"
OUT_DIR  = os.path.join(BASE_DIR, "Adv_graphRAG")
GRAPH_PATH = next((p for p in ["/content/drive/MyDrive/Research_kgTxagent/Kg_dev_files_and_data/test_openfda/OpenFDA_graph.pkl",
                               os.path.join(BASE_DIR, "OpenFDA_graph.pkl")] if os.path.isfile(p)), None)
assert GRAPH_PATH, "OpenFDA_graph.pkl not found"
chunk_files = sorted(glob.glob(os.path.join(OUT_DIR, "chunks_*.pkl")), key=os.path.getmtime)
assert chunk_files, "no chunks_*.pkl in " + OUT_DIR
CHUNK_PATH = chunk_files[-1]
SIG = os.path.basename(CHUNK_PATH)[len("chunks_"):-len(".pkl")]
SHARD_DIR = os.path.join(OUT_DIR, f"shards_{SIG}"); FAISS_PATH = os.path.join(OUT_DIR, f"faiss_{SIG}.index")
QA_PATH = os.path.join(BASE_DIR, "QA.json")
RESULTS_PATH = os.path.join(OUT_DIR, "qa_results_v4.json")
assert os.path.isdir(SHARD_DIR) and os.path.isfile(QA_PATH), "shards or QA.json missing"

ANSWER_MODEL = "labeltune-8b"      # CHANGED: tuned model writes the answers
AUX_MODEL    = "llama3:8b"         # CHANGED: base model does split + verify, as in V3
EMBED_MODEL  = "nomic-embed-text"
EMBED_URL, GEN_URL = "http://localhost:11434/api/embed", "http://localhost:11434/api/generate"
RERANK_MODEL, RERANK_DEVICE = "BAAI/bge-reranker-base", "cpu"

FLAGS = {"alias": True, "rerank": True, "split": "llm", "few_shot": False, "verify": True}   # CHANGED: few_shot off (optional)
NUM_CTX, ANSWER_TOKENS = 8192, 1200   # CHANGED: 700 -> 1200 so the Reasoning block cannot starve the answer
CONTEXT_TOKENS, CHARS_PER_TOKEN = 4500, 4
POOL, TOP_PASS = 15, 8
MAX_NAMED, PER_NAME = 4, 2
SEARCH_K, RERANK_N = 60, 60
CLUSTER_GAP = 2.0
PRECISION_MIN, ALIAS_MAX, STOP_DF = 0.15, 3, 300
EXCLUDE_SECTIONS = {"inactive_ingredient"}
CORE_SECTIONS = ["indications_and_usage", "warnings", "do_not_use", "pregnancy_or_breast_feeding", "stop_use"]
RRF_K, MAX_CHUNKS_PER_SECTION, ALIAS_CAP = 60, 2, 2
ABSTAIN_TEXT = "INSUFFICIENT FDA DATA: The retrieved label text does not contain this information."

# CHANGED: exact system prompt used in fine-tuning
SYSTEM_PROMPT = ("You answer ONLY from the FDA drug label excerpts below.\nRULES:\n"
    "1. Use only the excerpts. Never use outside knowledge.\n"
    "2. Write short bullet points. Every bullet ends with its source tag(s), e.g. [S2] or [S1][S3].\n"
    "3. Keep drugs separate: start each bullet with the drug name.\n"
    f"4. If the excerpts do not answer the question, reply exactly: {ABSTAIN_TEXT}\n"
    "5. Give no medical advice beyond what the label says.")
STOPS = ["\n\nQUESTION:", "\n\nEXCERPTS:", "<|eot_id|>", "<|end_of_text|>"]   # CHANGED: stop the model before it echoes the prompt
print("SIG:", SIG, "| answer model:", ANSWER_MODEL, "| aux model:", AUX_MODEL)

SIG: 7c9ff259 | answer model: labeltune-8b | aux model: llama3:8b


In [5]:
# ===== 3. load graph, chunks, embeddings, FAISS, BM25 =====
t0 = time.time()
with open(GRAPH_PATH, "rb") as f: G = pickle.load(f)
type_counts = Counter(d.get("type") for _, d in G.nodes(data=True))
SECTION_TYPES = sorted(t for t in type_counts if t not in ("drug_generic", "drug_brand"))
generic_ids = [n for n, d in G.nodes(data=True) if d.get("type") == "drug_generic"]
brand_ids   = [n for n, d in G.nodes(data=True) if d.get("type") == "drug_brand"]
GENERIC_NAME = {n: G.nodes[n].get("name", "") for n in generic_ids}
BRAND_NAME   = {n: G.nodes[n].get("name", "") for n in brand_ids}
BRAND_TO_GENERIC, DRUG_SECTIONS = defaultdict(list), defaultdict(list)
for g in tqdm(generic_ids, desc="drug->section edges"):      # only drug->section edges
    for nb in G.neighbors(g):
        t = G.nodes[nb].get("type")
        if t == "drug_brand": BRAND_TO_GENERIC[nb].append(g)
        elif t in SECTION_TYPES: DRUG_SECTIONS[g].append((nb, t))
print(f"graph {time.time()-t0:.0f}s | generics={len(generic_ids):,} brands={len(brand_ids):,}")

def section_text(sid):
    txt = G.nodes[sid].get("description") or ""
    return re.sub(r"\s+", " ", html.unescape(re.sub(r"<[^>]+>", " ", txt))).strip()

CHUNKS = pickle.load(open(CHUNK_PATH, "rb"))
EMB = np.ascontiguousarray(np.vstack([np.load(p) for p in sorted(glob.glob(os.path.join(SHARD_DIR, "emb_*.npy")))]), dtype="float32")
assert EMB.shape[0] == len(CHUNKS) and np.isfinite(EMB).all(), "embeddings do not match chunks"
faiss.normalize_L2(EMB)
INDEX = faiss.read_index(FAISS_PATH) if os.path.isfile(FAISS_PATH) else None
if INDEX is None or INDEX.ntotal != EMB.shape[0]:
    INDEX = faiss.IndexFlatIP(EMB.shape[1]); INDEX.add(EMB)

STOP = {"the","a","an","of","and","to","is","are","can","i","if","for","in","on","or","with","what","do","how","my","me","it"}
def tokenize(s):
    s = re.sub(r"(\d)([a-z])", r"\1 \2", s.lower())
    return [t for t in re.findall(r"[a-z0-9]+", s) if t not in STOP]
BM25 = bm25s.BM25(); BM25.index([tokenize(c["stamped"]) for c in CHUNKS], show_progress=False)
IDX_BY_SECTION, IDX_BY_DRUG = defaultdict(list), defaultdict(list)
for i, c in enumerate(CHUNKS):
    IDX_BY_SECTION[c["section"]].append(i); IDX_BY_DRUG[c["drug_id"]].append(i)
IDX_BY_SECTION = {k: np.array(v) for k, v in IDX_BY_SECTION.items()}
IDX_BY_DRUG = {k: np.array(v) for k, v in IDX_BY_DRUG.items()}
print("chunks:", len(CHUNKS), "| EMB:", EMB.shape, "| FAISS:", INDEX.ntotal)

drug->section edges:   0%|          | 0/10840 [00:00<?, ?it/s]

graph 3s | generics=10,840 brands=31,797


DEBUG:bm25s:Building index from tokens


chunks: 97579 | EMB: (97579, 768) | FAISS: 97579


In [6]:
# ===== 4. LLM + embedding helpers, hybrid search =====
def _embed_call(texts):
    r = requests.post(EMBED_URL, json={"model": EMBED_MODEL, "input": list(texts), "truncate": True}, timeout=300)
    r.raise_for_status()
    arr = np.asarray(r.json()["embeddings"], dtype="float32")
    assert arr.ndim == 2 and arr.shape[1] == EMB.shape[1], f"embedding shape {arr.shape} vs cache {EMB.shape}"
    return arr.tolist()

_QCACHE = {}
def qvec(q):                                    # CHANGED: always a flat 768-vector (fixes the (1,768) crash)
    if q not in _QCACHE:
        v = np.asarray(_embed_call(["search_query: " + q])[0], dtype="float32").reshape(-1)
        _QCACHE[q] = v / (np.linalg.norm(v) + 1e-9)
    return _QCACHE[q]

META = {"prompt_tokens": 0, "done_reason": ""}
def llm(prompt, model=None, system=None, temperature=0.1, num_predict=500, json_mode=False, stop=None, tries=3):
    p = {"model": model or AUX_MODEL, "prompt": prompt, "stream": False, "keep_alive": "30m",
         "options": {"temperature": temperature, "num_ctx": NUM_CTX, "num_predict": int(num_predict)}}
    if system: p["system"] = system
    if stop: p["options"]["stop"] = stop
    if json_mode: p["format"] = "json"
    err = None
    for k in range(tries):
        try:
            r = requests.post(GEN_URL, json=p, timeout=900); r.raise_for_status(); j = r.json()
            META["prompt_tokens"] = int(j.get("prompt_eval_count", 0) or 0); META["done_reason"] = j.get("done_reason", "")
            return j["response"].strip()
        except Exception as e:
            err = e; print(f"  llm retry {k+1}/{tries}: {repr(e)[:100]}"); time.sleep(4 * (k + 1))
    raise RuntimeError(f"Ollama call failed: {err!r}")

def llm_json(prompt, **kw):
    raw = llm(prompt, json_mode=True, num_predict=200, **kw)
    try: return json.loads(raw)
    except Exception:
        m = re.search(r"\{.*\}", raw, re.S)
        try: return json.loads(m.group(0)) if m else None
        except Exception: return None

def dense_topk(qv, allowed, k):
    if allowed is None:
        _, I = INDEX.search(qv.reshape(1, -1).astype("float32"), k); return [int(i) for i in I[0] if i >= 0]
    sims = EMB[allowed] @ qv
    return [int(allowed[o]) for o in np.argsort(-sims)[:k]]
def bm25_topk(q, allowed, k):
    toks = tokenize(q)
    if not toks: return []
    scores = BM25.get_scores(toks)
    if allowed is None:
        cand = np.argpartition(-scores, min(k, len(scores) - 1))[:k]; cand = cand[np.argsort(-scores[cand])]
    else: cand = allowed[np.argsort(-scores[allowed])[:k]]
    return [int(c) for c in cand if scores[c] > 0]
def hybrid_rank(queries, allowed=None, k=50):
    out = defaultdict(float)
    for q in queries:
        for lst in (dense_topk(qvec(q), allowed, k), bm25_topk(q, allowed, k)):
            for rank, idx in enumerate(lst, 1): out[idx] += 1.0 / (RRF_K + rank)
    return sorted(out.items(), key=lambda x: -x[1])[:k]

RERANKER = None
def get_reranker():
    global RERANKER
    if RERANKER is None:
        from sentence_transformers import CrossEncoder
        RERANKER = CrossEncoder(RERANK_MODEL, max_length=512, device=RERANK_DEVICE)
    return RERANKER

print("aux model :", llm("Reply with the single word OK.", model=AUX_MODEL, num_predict=5, temperature=0))
print("tuned model:", llm("EXCERPTS:\n[S1] (warnings) For external use only\n\nQUESTION: Is it for external use?\n\nANSWER:\n",
                          model=ANSWER_MODEL, system=SYSTEM_PROMPT, num_predict=120, stop=STOPS))
print("query vector shape:", qvec("test").shape)

aux model : OK
tuned model: Reasoning: Okay, let's think about this. I need to figure out if this product is meant for external use or not. So, what do I know about external use? Well, usually, when something is for external use, it's meant to be applied to the skin or maybe even the hair. It's not supposed to be taken internally, like swallowed or injected. Now, let's look at the product. What does it say on the label? Hmm, I should check the label carefully. Oh, here it is! It says 'For external use only.' That's pretty
query vector shape: (768,)


In [7]:
# ===== 5. entity table + alias table (identical to V3) =====
def norm_tokens(s):
    s = re.sub(r"(\d)([a-z])", r"\1 \2", s.lower())
    return [t for t in re.findall(r"[a-z0-9]+", s) if t not in {"and", "the", "of", "with"}]
squash = lambda s: re.sub(r"[^a-z0-9]", "", s.lower())

name_vocab = defaultdict(set)
for g, nm in GENERIC_NAME.items():
    for t in set(norm_tokens(nm)): name_vocab[t].add(g)
for b, nm in BRAND_NAME.items():
    for t in set(norm_tokens(nm)): name_vocab[t].update(BRAND_TO_GENERIC.get(b, []))
drug_tokens = defaultdict(set)
for c in tqdm(CHUNKS, desc="tokens per drug"): drug_tokens[c["drug_id"]].update(norm_tokens(c["text"]))
DF = Counter()
for d, ts in drug_tokens.items(): DF.update(ts)
text_vocab = defaultdict(set)
for d, ts in drug_tokens.items():
    for t in ts & name_vocab.keys(): text_vocab[t].add(d)
TOK_PREC = {t: (len(nd & text_vocab[t]) / len(text_vocab[t])) if text_vocab.get(t) else 1.0 for t, nd in name_vocab.items()}
ENT, dropped = [], []
for kind, table in (("generic", GENERIC_NAME), ("brand", BRAND_NAME)):
    for eid, nm in table.items():
        if kind == "brand" and not BRAND_TO_GENERIC.get(eid): continue
        ts = frozenset(norm_tokens(nm))
        if not ts: continue
        (ENT if max(TOK_PREC[t] for t in ts) >= PRECISION_MIN else dropped).append((ts, nm, kind, eid))
print(f"entities kept={len(ENT):,} | dropped as category words={len(dropped):,}")

QSTOP = {"compare","versus","difference","different","products","product","safe","use","used","using","what","which",
         "depending","whether","between","child","children","pregnant","pregnancy","warning","warnings"}
STOP_DATA = {t for t, c in DF.items() if c > STOP_DF}
dom_re = re.compile(r"\b([A-Za-z0-9][A-Za-z0-9\-]{2,})\.(?:com|net|org|co)\b")
tm_re  = re.compile(r"\b([A-Z][A-Za-z0-9\-]{3,})\s?(?:\u2122|\u00ae|\(TM\)|TM\b)")
cap_re = re.compile(r"\b[A-Z][a-z]{4,}[A-Za-z\-]*\b")
alias_dom, cap_hits = defaultdict(set), defaultdict(set)
for c in tqdm(CHUNKS, desc="alias scan"):
    d, t = c["drug_id"], c["text"]
    for m in dom_re.findall(t) + tm_re.findall(t): alias_dom[squash(m)].add(d)
    for m in set(cap_re.findall(t)): cap_hits[squash(m)].add(d)
BAD = QSTOP | STOP_DATA
ALIAS = defaultdict(set)
for a, ds in alias_dom.items():
    if 4 <= len(a) and len(ds) <= ALIAS_MAX and a not in BAD: ALIAS[a] |= ds
for a, ds in cap_hits.items():
    if len(a) >= 5 and len(ds) <= ALIAS_MAX and a not in name_vocab and a not in BAD and DF.get(a, 0) <= 6: ALIAS[a] |= ds
print("alias entries:", len(ALIAS))

def alias_hits(query):
    toks = [squash(t) for t in re.split(r"[\s/,?;:]+", query) if t]
    cand = {t for t in toks if len(t) >= 4} | {a + b for a, b in zip(toks, toks[1:]) if a and b}
    return [{"drug_id": g, "matched": f"alias:{a}", "n": 99} for a in cand if a not in BAD for g in ALIAS.get(a, ())]

def detect_entities(query):
    qn = " ".join(norm_tokens(query)); qt = set(qn.split())
    exact, fuzzy = [], []
    for ts, nm, kind, eid in ENT:
        if ts <= qt: exact.append((ts, nm, kind, eid))
        elif len(ts) >= 5 and len(ts) - len(ts & qt) == 1: fuzzy.append((ts, nm, kind, eid))
    hits = exact if exact else sorted(fuzzy, key=lambda h: -len(h[0]))[:1]
    qp = " " + qn + " "
    def spans(nm):
        ph, out, s = " " + " ".join(norm_tokens(nm)) + " ", [], 0
        while True:
            i = qp.find(ph, s)
            if i < 0: return out
            out.append((i, i + len(ph))); s = i + 1
    sp = {h[1]: spans(h[1]) for h in hits}
    keep = []
    for h in hits:
        longer = [o for o in hits if h[0] < o[0]]
        if not longer: keep.append(h); continue
        cover = [x for o in longer for x in sp[o[1]]]
        if [x for x in sp[h[1]] if not any(a <= x[0] and x[1] <= b for a, b in cover)]: keep.append(h)
    out = []
    for ts, nm, kind, eid in sorted(keep, key=lambda h: (-len(h[0]), len(h[1]))):
        for g in ([eid] if kind == "generic" else BRAND_TO_GENERIC.get(eid, [])):
            out.append({"drug_id": g, "matched": nm, "n": len(ts), "kind": "exact" if ts <= qt else "fuzzy"})
    if FLAGS["alias"]:
        skip = set().union(*[h[0] for h in keep]) if keep else set()
        al, seen_al = [], set()
        for h in alias_hits(query):
            if h["matched"][6:] in skip or h["drug_id"] in seen_al: continue
            seen_al.add(h["drug_id"]); al.append(dict(h, kind="alias"))
        out += al[:ALIAS_CAP]
    return out

_LH = {}
def label_hash(d):
    if d not in _LH:
        txt = "||".join(sorted(section_text(sid).lower() for sid, t in DRUG_SECTIONS[d] if t not in EXCLUDE_SECTIONS))
        _LH[d] = hashlib.md5(txt.encode()).hexdigest()
    return _LH[d]
LAST_HITS = []
def named_drugs(query):
    per, order, seen_h = Counter(), [], set()
    hs = detect_entities(query)
    LAST_HITS.clear(); LAST_HITS.extend((h["matched"], h["kind"], GENERIC_NAME[h["drug_id"]]) for h in hs[:12])
    for h in hs:
        if per[h["matched"]] >= PER_NAME: continue
        hh = label_hash(h["drug_id"])
        if hh in seen_h or h["drug_id"] in order: continue
        seen_h.add(hh); per[h["matched"]] += 1; order.append(h["drug_id"])
        if len(order) >= MAX_NAMED: break
    return order
print("test:", [GENERIC_NAME[d] for d in named_drugs("Can I use Kank-a on my 1-year-old's gums for teething?")])

tokens per drug:   0%|          | 0/97579 [00:00<?, ?it/s]

entities kept=39,939 | dropped as category words=2,695


alias scan:   0%|          | 0/97579 [00:00<?, ?it/s]

alias entries: 3127
test: ['BENZOCAINE AND MENTHOL, UNSPECIFIED FORM']


In [8]:
# ===== 6. section choice (embeddings) + candidate search, rerank, meaning split (identical to V3) =====
SECTION_DESC = {
 "indications_and_usage": "what the product is used for, uses, relieves, treats symptoms",
 "purpose": "purpose of the product", "active_ingredient": "active ingredient and strength",
 "dosage_and_administration": "how much and how often to take or apply, directions, dose by age",
 "dosage_and_administration_table": "dosage table by age or weight",
 "warnings": "warnings, risks, side effects, allergy alert",
 "do_not_use": "do not use, who should not use this product, contraindications",
 "ask_doctor": "ask a doctor before use if you have a medical condition",
 "ask_doctor_or_pharmacist": "ask a doctor or pharmacist before use if you take other drugs",
 "stop_use": "when to stop use and ask a doctor, symptoms last more than days",
 "pregnancy_or_breast_feeding": "pregnant or breast-feeding, ask a health professional",
 "keep_out_of_reach_of_children": "keep out of reach of children, overdose, poison control",
 "storage_and_handling": "how to store the product, storage temperature"}
SEC_LIST = [s for s in SECTION_TYPES if s not in EXCLUDE_SECTIONS]
assert all(c in SECTION_TYPES for c in CORE_SECTIONS)
SEC_EMB = np.array(_embed_call(["search_document: " + SECTION_DESC.get(s, s.replace("_", " ")) for s in SEC_LIST]), dtype="float32")
SEC_EMB /= (np.linalg.norm(SEC_EMB, axis=1, keepdims=True) + 1e-9)
SECTION_ORDER = ["indications_and_usage","purpose","active_ingredient","dosage_and_administration","dosage_and_administration_table",
                 "warnings","do_not_use","ask_doctor","ask_doctor_or_pharmacist","stop_use","pregnancy_or_breast_feeding",
                 "keep_out_of_reach_of_children","storage_and_handling"]
def choose_sections(q, top=3):
    sc = dict(zip(SEC_LIST, SEC_EMB @ qvec(q)))
    ranked = sorted(sc, key=lambda s: -sc[s])
    wanted = set(CORE_SECTIONS) | {s for s in ranked[:top] if s != "storage_and_handling" or sc[s] >= 0.6}
    return wanted, ranked[:top]

def search_query(q, extra=()):
    idxA = np.concatenate([IDX_BY_SECTION[s] for s in ("indications_and_usage", "purpose") if s in IDX_BY_SECTION])
    fused = defaultdict(float)
    for i, s in hybrid_rank([q], idxA, k=SEARCH_K): fused[i] += s
    for i, s in hybrid_rank([q], None, k=15): fused[i] += 0.5 * s
    cand = sorted(fused, key=lambda i: -fused[i])[:RERANK_N]
    cand += [i for i in extra if i not in cand]
    if FLAGS["rerank"] and cand:
        sc = get_reranker().predict([(q, CHUNKS[i]["stamped"]) for i in cand], show_progress_bar=False)
        scores = {i: float(s) for i, s in zip(cand, sc)}
    else: scores = {i: fused.get(i, 0.0) for i in cand}
    return sorted(cand, key=lambda i: -scores[i]), scores

def drugs_from(cand, scores, n):
    best = {}
    for i in cand: best.setdefault(CHUNKS[i]["drug_id"], scores[i])
    out, seen = [], set()
    for d, s in sorted(best.items(), key=lambda x: -x[1]):
        h = label_hash(d)
        if h in seen: continue
        seen.add(h); out.append(d)
        if len(out) >= n: break
    return out

def llm_split(q):
    p = ("Return JSON only. If the question asks about two or three DIFFERENT things (two meanings of one word, or two conditions), "
         "give one short search phrase per thing. Each phrase must keep the main topic words of the question. "
         "If it asks about one thing, return an empty list.\nFormat: {\"phrases\": [\"...\"]}\nQuestion: " + q)
    r = llm_json(p)
    try: ph = [x.strip() for x in r["phrases"] if isinstance(x, str) and x.strip()]
    except Exception: return []
    return ph if 2 <= len(ph) <= 3 else []

def cluster_groups(cand, scores):
    top = cand[:30]
    if len(top) < 8: return None
    from sklearn.cluster import KMeans
    lab = KMeans(2, n_init=5, random_state=0).fit(EMB[top]).labels_
    groups = [[i for i, l in zip(top, lab) if l == k] for k in (0, 1)]
    if min(len(g) for g in groups) < 3: return None
    if abs(max(scores[i] for i in groups[0]) - max(scores[i] for i in groups[1])) > CLUSTER_GAP: return None
    ds = [{CHUNKS[i]["drug_id"] for i in g} for g in groups]
    if len(ds[0] & ds[1]) / max(1, min(len(ds[0]), len(ds[1]))) > 0.5: return None
    return groups

def symptom_groups(q):
    phrases = llm_split(q) if FLAGS["split"] == "llm" else []
    cand0, sc0 = search_query(q)
    if phrases:
        gs = []
        for ph in phrases:
            cand, sc = search_query(ph, extra=cand0[:20]); gs.append({"label": ph, "drugs": drugs_from(cand, sc, POOL // len(phrases))})
        return gs, "llm-split"
    if FLAGS["split"] in ("llm", "cluster"):
        cl = cluster_groups(cand0, sc0)
        if cl: return [{"label": f"cluster{k}", "drugs": drugs_from(g, sc0, POOL // 2)} for k, g in enumerate(cl)], "cluster"
    return [{"label": "main", "drugs": drugs_from(cand0, sc0, POOL)}], "single"
print("ready")

ready


In [9]:
# ===== 7. context blocks + prompt in the TRAINING format =====
SECTION_RANK = {s: i for i, s in enumerate(SECTION_ORDER)}
def build_block(d, wanted, missing_note, qvs, quota, tagc, sources):
    idxs = [i for i in IDX_BY_DRUG.get(d, []) if CHUNKS[i]["section"] in wanted]
    idxs = sorted(idxs, key=lambda i: -max(float(EMB[i] @ v) for v in qvs))
    chosen, used, seen = [], 0, set()
    for i in idxs:
        s = CHUNKS[i]["section"]
        if s not in seen and used + len(CHUNKS[i]["text"]) <= quota: chosen.append(i); seen.add(s); used += len(CHUNKS[i]["text"])
    per_sec = Counter(CHUNKS[i]["section"] for i in chosen)
    for i in idxs:
        sec = CHUNKS[i]["section"]
        if i not in chosen and per_sec[sec] < MAX_CHUNKS_PER_SECTION and used + len(CHUNKS[i]["text"]) <= quota:
            chosen.append(i); used += len(CHUNKS[i]["text"]); per_sec[sec] += 1
    chosen.sort(key=lambda i: (SECTION_RANK.get(CHUNKS[i]["section"], 99), i))
    have = {CHUNKS[i]["section"] for i in IDX_BY_DRUG.get(d, [])}
    lines = [f"### {GENERIC_NAME[d]}"]
    miss = [s for s in missing_note if s not in have]
    if miss: lines.append("NOTE: no text for " + ", ".join(miss) + ".")
    for i in chosen:
        tagc[0] += 1; tag = f"S{tagc[0]}"
        sources.append({"tag": tag, "drug_id": d, "drug": GENERIC_NAME[d], "section": CHUNKS[i]["section"], "node": CHUNKS[i]["cid"], "text": CHUNKS[i]["text"]})
        lines.append(f"[{tag}] ({CHUNKS[i]['section']}) {CHUNKS[i]['text']}")
    return "\n".join(lines)

def order_drugs(ds):
    return [ds[0]] + ds[2:] + [ds[1]] if len(ds) >= 6 else ds     # same as V3

def build_context(q, groups_drugs, shrink=1.0, labels=None):
    wanted, top3 = choose_sections(q); qv = qvec(q)
    total = int(CONTEXT_TOKENS * CHARS_PER_TOKEN * shrink)
    live = [(g, lb) for g, lb in zip(groups_drugs, labels or [None] * len(groups_drugs)) if g]
    group_budget = total // max(1, len(live))
    tagc, sources, blocks = [0], [], []
    for g, lb in live:
        quota = min(group_budget // len(g), 9000)
        qvs = [qv] + ([qvec(lb)] if lb else [])
        for d in order_drugs(g): blocks.append(build_block(d, wanted, top3, qvs, quota, tagc, sources))
    return "\n\n".join(blocks), sources

# optional one-shot example, written in the trained format (off by default)
FEW_SHOT = ("EXCERPTS:\n### EXAMPLEDRUG\n[S1] (warnings) Do not use if seal is broken ; Stop use and ask a doctor if rash occurs\n\n"
            "QUESTION: When should I stop using exampledrug?\n\nANSWER:\n"
            "Reasoning: The warnings say to stop if a rash occurs.\n\n- Exampledrug: stop use and ask a doctor if rash occurs [S1]\n\n")

def build_prompt(q, context):
    # CHANGED: user message is exactly EXCERPTS / QUESTION / ANSWER (system prompt is sent separately)
    return (FEW_SHOT if FLAGS["few_shot"] else "") + f"EXCERPTS:\n{context}\n\nQUESTION: {q}\n\nANSWER:\n"

In [10]:
# ===== 8. parse the tuned model's output, answer, verify =====
TAG_RE = r"\[S(\d+)(?:\s*[-\u2013]\s*S?(\d+))?\]"
def get_tags(text):
    tags = []
    for a, b in re.findall(TAG_RE, text): tags += [f"S{n}" for n in range(int(a), int(b) + 1)] if b else [f"S{a}"]
    return list(dict.fromkeys(tags))

def split_reasoning(raw):
    """Model writes 'Reasoning: <thinking>' + blank line + '- bullet [S1]'. Return (reasoning, answer_text)."""
    raw = raw.strip()
    m = re.match(r"(?is)^\s*Reasoning:\s*(.*)$", raw)
    if not m: return "", raw
    rest = m.group(1)
    b = re.search(r"\n\s*\n(?=\s*[-*\u2022]\s)", rest) or re.search(r"\n\s*\n(?=\s*INSUFFICIENT FDA DATA)", rest)
    if b: return rest[:b.start()].strip(), rest[b.end():].strip()
    b = re.search(r"(?m)^\s*[-*\u2022]\s", rest)
    if b: return rest[:b.start()].strip(), rest[b.start():].strip()
    i = rest.find(ABSTAIN_TEXT[:20])
    if i >= 0: return rest[:i].strip(), rest[i:].strip()
    return rest.strip(), ""                      # no answer written (cut off or off-format)

def parse_answer(raw):
    reasoning, ans = split_reasoning(raw)
    claims = []
    for line in ans.splitlines():
        line = line.strip()
        if not line: continue
        if ABSTAIN_TEXT[:20] in line:
            claims.append({"claim": ABSTAIN_TEXT, "tags": [], "gap": True}); continue
        tags = get_tags(line)
        if not (line[0] in "-*\u2022" or tags): continue         # reasoning text is never a claim
        text = line.lstrip("-*\u2022 ").strip()
        claims.append({"claim": re.sub(r"\[S[\d\s\-\u2013S]*\]", "", text).strip(), "tags": tags, "gap": text.lower().startswith("gap:")})
    return reasoning, claims

CSTOP = STOP | {"this","that","from","have","with","text","label","retrieved","product","use","used"}
def overlap(claim, ev):
    cw = {w for w in re.findall(r"[a-z0-9]+", claim.lower()) if len(w) >= 4 and w not in CSTOP}
    ew = set(re.findall(r"[a-z0-9]+", ev.lower()))
    return (len(cw & ew) / len(cw)) if cw else 1.0

def verify(claims, sources):
    by = {s["tag"]: s for s in sources}
    for c in claims:
        if c["gap"]: c["status"] = "gap"; continue
        if not c["tags"]: c["status"] = "no_citation"; continue
        if any(t not in by for t in c["tags"]): c["status"] = "invalid_citation"; continue
        ev = "\n".join(by[t]["text"] for t in c["tags"])
        if overlap(c["claim"], ev) >= 0.8: c["status"] = "supported"; continue
        v = llm(f"Label text:\n{ev}\n\nClaim: {c['claim']}\n\nDoes the label text directly support the claim? Answer with one word: YES or NO.",
                model=AUX_MODEL, temperature=0.0, num_predict=3)
        c["status"] = "supported" if v.upper().startswith("YES") else "unsupported"
    return claims

def generate(user):
    raw = llm(user, model=ANSWER_MODEL, system=SYSTEM_PROMPT, num_predict=ANSWER_TOKENS, stop=STOPS)
    ptoks, retried = META["prompt_tokens"], False
    if META["done_reason"] == "length" and not split_reasoning(raw)[1]:      # reasoning ate the budget: retry once with more room
        room = NUM_CTX - ptoks - 50
        if room > ANSWER_TOKENS:
            raw = llm(user, model=ANSWER_MODEL, system=SYSTEM_PROMPT, num_predict=min(ANSWER_TOKENS * 2, room), stop=STOPS); retried = True
    return raw, ptoks, retried, META["done_reason"] == "length"

def answer_question(q, verbose=True):
    t0 = time.time(); named = named_drugs(q); hits = list(LAST_HITS)
    if named: mode, groups, how = "drug", [{"label": "named", "drugs": named}], "named"
    else: groups, how = symptom_groups(q); mode = "symptom"
    k = max(1, TOP_PASS // len(groups)); groups_drugs = [g["drugs"][:k] for g in groups] if not named else [named]
    if not any(groups_drugs):
        return {"question": q, "mode": mode, "drugs": [], "sources": [], "claims": [], "abstained": True, "abstain_reason": "no chunks retrieved",
                "final_answer": ABSTAIN_TEXT, "raw_answer": ABSTAIN_TEXT, "seconds": round(time.time() - t0, 1), "sections_in_context": []}
    shrink = 1.0
    while True:
        context, sources = build_context(q, groups_drugs, shrink, [g["label"] if how == "llm-split" else None for g in groups])
        user = build_prompt(q, context)
        if (len(user) + len(SYSTEM_PROMPT)) / 3.5 + ANSWER_TOKENS <= NUM_CTX or shrink < 0.3: break
        shrink *= 0.8
    raw, ptoks, retried, truncated = generate(user)
    reasoning, claims = parse_answer(raw)
    if FLAGS["verify"]: claims = verify(claims, sources)
    real = [c for c in claims if not c["gap"]]
    rate = round(sum(c.get("status") != "supported" for c in real) / len(real), 3) if real else None
    if claims:
        final = "\n".join(("- " if c.get("status") in ("supported", "gap", None) else f"- [FLAGGED: {c['status']}] ") + c["claim"] + " " + "".join(f"[{t}]" for t in c["tags"]) for c in claims)
    else: final = "(no bullet answer produced, see raw_answer)"
    reason = None if real else ("answer has only Gap/abstain lines" if claims else "no answer parsed (cut off or off-format)")
    rec = {"question": q, "mode": mode, "split": how, "entity_hits": hits,
           "groups": [{"label": g["label"], "drugs": [GENERIC_NAME[d] for d in g["drugs"]]} for g in groups],
           "drugs": [GENERIC_NAME[d] for g in groups_drugs for d in g], "sections_in_context": sorted({s["section"] for s in sources}),
           "sources": [{k2: v for k2, v in s.items() if k2 not in ("text", "drug_id")} for s in sources], "context_chars": len(context),
           "prompt_tokens": ptoks, "reasoning": reasoning, "retried_longer": retried, "truncated": truncated, "claims": claims,
           "unsupported_rate": rate, "raw_answer": raw, "final_answer": final, "abstained": not real, "abstain_reason": reason,
           "seconds": round(time.time() - t0, 1)}
    if verbose: print(f"\nQ: {q}\n mode={mode}/{how} drugs={rec['drugs']} prompt_tokens={ptoks} truncated={truncated}\n{final}\n ({rec['seconds']}s)")
    return rec
print("pipeline ready")

pipeline ready


In [11]:
# ===== 9. load QA + entity check (no LLM) =====
QA = json.load(open(QA_PATH))
norm_name = lambda s: re.sub(r"[^A-Z0-9]+", " ", s.upper()).strip()
gold_drugs = lambda qa: [m for t in qa["trace"] for m in re.findall(r"DRUG: (.*?) \| SOURCE", t)]
rows = []
for qid, qa in QA.items():
    got = {norm_name(GENERIC_NAME[d]) for d in named_drugs(qa["question"])}
    gold = [norm_name(g) for g in gold_drugs(qa)]
    rows.append({"qid": qid, "named_found": len(got), "gold_hit": sum(g in got for g in gold), "gold_n": len(gold)})
ec = pd.DataFrame(rows); ec["type"] = ec.qid.str.split("-").str[0]
print(ec.groupby("type").agg(n=("qid", "count"), named_found=("named_found", "mean"), gold_hit=("gold_hit", "sum"), gold_n=("gold_n", "sum")).round(2))

      n  named_found  gold_hit  gold_n
type                                  
T1    6         2.00         4       6
T2    8         1.62         0      11
T3    9         1.22         6       9
T4    9         0.78         0      18
T5    9         3.22        14      18
T6    9         1.33         9       9
T7    5         2.40         5       5
T8    5         2.40         4       8


In [12]:
# ===== 10. PROBE: 5 questions. Check format BEFORE the full run =====
for qid in ["T1-001", "T5-004", "T6-003", "T4-001", "T3-006"]:
    rec = answer_question(QA[qid]["question"], verbose=False)
    print("=" * 70, "\n", qid, QA[qid]["question"])
    print("RAW (first 600):\n", rec["raw_answer"][:600])
    print("-> starts with Reasoning:", rec["raw_answer"].lstrip().startswith("Reasoning:"), "| truncated:", rec["truncated"],
          "| retried:", rec["retried_longer"], "| claims:", len(rec["claims"]), "| abstained:", rec["abstained"])
    print("FINAL:\n", rec["final_answer"]); print("KEY:", QA[qid]["answer"][:300])

 T1-001 What should I avoid if I am pregnant and using lidocaine?
RAW (first 600):
 Reasoning: Okay, so I'm thinking about lidocaine and pregnancy. First off, I know that lidocaine is a local anesthetic, which means it numbs a specific area. It's usually used for things like dental procedures or to help with pain relief. But, I'm curious about how it might affect someone who is pregnant. Hmm, I remember that during pregnancy, the body's systems can be a bit more sensitive, especially when it comes to medications. So, I wonder if lidocaine could cross the placenta and affect the baby. That's definitely something to be cautious about. Oh, and I've heard that lidocaine can cause
-> starts with Reasoning: True | truncated: False | retried: False | claims: 1 | abstained: False
FINAL:
 - (pregnancy_or_breast_feeding) If pregnant or breast-feeding, ask a health professional before use. [S4]
KEY: D0337 (ALLANTOIN, LIDOCAINE): If pregnant or breast-feeding, ask a health professional before use.

In [13]:
# ===== 11. T8 check: an injected unsupported sentence must NOT be marked supported =====
q = QA["T3-005"]["question"]; named = named_drugs(q)
ctx, src = build_context(q, [named])
real = {"claim": GENERIC_NAME[named[0]] + ": " + src[0]["text"][:140], "tags": ["S1"], "gap": False}
fake = {"claim": "The ointment is safe for children over 6 years of age.", "tags": ["S1"], "gap": False}
out = verify([real, fake], src)
for c in out: print(c["status"].upper(), "|", c["claim"][:90])
print("PASS" if out[1]["status"] != "supported" else "FAIL: injected claim accepted")

SUPPORTED | DICLOFENAC SODIUM, METHYL SALICYLATE: 1. ​INDICATIONS & USAGE Xiclofen TM Ointment is indi
UNSUPPORTED | The ointment is safe for children over 6 years of age.
PASS


In [14]:
# ===== 12. FULL RUN (resumable, saved after every question) =====
RESET = True      # True: back up an old qa_results_v4.json and start fresh. Set False to resume after a restart.
if RESET and "RESET_DONE" not in globals() and os.path.exists(RESULTS_PATH):
    os.rename(RESULTS_PATH, RESULTS_PATH.replace(".json", f"_old_{int(time.time())}.json"))
RESET_DONE = True
SKIP_WORDS = {"and","the","with","mg","unspecified","form","tablet","tablets","cream","ointment","gel","solution","topical","oral","liquid"}
ing = lambda s: frozenset(t for t in re.findall(r"[a-z]{3,}", s.lower()) if t not in SKIP_WORDS)
RESULTS = json.load(open(RESULTS_PATH)) if os.path.exists(RESULTS_PATH) else {}
todo = [q for q in QA if q not in RESULTS]; print(f"done {len(RESULTS)} | to run {len(todo)}")
for n, qid in enumerate(todo, 1):
    qa = QA[qid]; print(f"\n######## {qid} ({n}/{len(todo)})")
    try: rec = answer_question(qa["question"])
    except Exception as e:
        print("!! ERROR", qid, repr(e)); traceback.print_exc(); continue
    gold_raw = gold_drugs(qa); got = [norm_name(d) for d in rec["drugs"]]; got_ing = {ing(d) for d in rec["drugs"]}
    rec["gold_drugs"] = [norm_name(g) for g in gold_raw]
    rec["drug_recall"] = round(sum(norm_name(g) in got for g in gold_raw) / len(gold_raw), 2) if gold_raw else None
    rec["drug_recall_ing"] = round(sum(ing(g) in got_ing for g in gold_raw) / len(gold_raw), 2) if gold_raw else None
    rec["section_recall"] = round(len(set(qa["sections"]) & set(rec["sections_in_context"])) / len(qa["sections"]), 2) if qa["sections"] else None
    rec["answerable"], rec["answer_key"] = qa["answerable"], qa["answer"]
    rec["abstain_correct"] = (rec["abstained"] == (not qa["answerable"]))
    RESULTS[qid] = rec; json.dump(RESULTS, open(RESULTS_PATH, "w"), indent=2)
print("saved", len(RESULTS), "->", RESULTS_PATH)

done 0 | to run 60

######## T1-001 (1/60)

Q: What should I avoid if I am pregnant and using lidocaine?
 mode=drug/named drugs=['LIDOCAINE', 'LIDOCAINE HYDROCHLORIDE', 'LIDOCAINE 4%'] prompt_tokens=899 truncated=False
- (pregnancy_or_breast_feeding) If pregnant or breast-feeding, ask a health professional before use. [S4]
 (5.9s)

######## T1-002 (2/60)

Q: What organ-specific warning and daily intake limit apply to Acetaminophen and Phenylephrine HCl, and what medical conditions require consulting a doctor before use?
 mode=drug/named drugs=['ACETAMINOPHEN PHENYLEPHRINE HCL', 'ACETAMINOPHEN AND PHENYLEPHRINE HCL'] prompt_tokens=1405 truncated=False
- Uses temporarily relieves these symptoms associated with hay fever or other upper respiratory allergies, and the common cold: minor aches and pains nasal congestion headache sinus congestion and pressure promotes sinus drainage temporarily reduces fever [S1]
 (7.5s)

######## T1-003 (3/60)

Q: What local anesthetic allergy warning and mo

config.json:   0%|          | 0.00/799 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.11GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

... (2 outputs left out)

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]


Q: What over-the-counter oral pain relievers are indicated for headaches, and what specific warnings or precautions apply to their use during pregnancy?
 mode=symptom/llm-split drugs=['DIVALPROEX SODIUM', 'VALPROIC ACID', 'ARNICA MONTANA, CAULOPHYLLUM THALICTROIDES, CIMICIFUGA RACEMOSA, GELSEMIUM SEMPERVIRENS, PULSATILLA.', 'APIS MEL., BERBER. VULG., BRYONIA, CANTHARIS, CINCHONA, EQUISETUM HYEMALE, FERRUM PHOSPHORICUM, GELSEMIUM, JUNIPERUS COM., PAREIRA, PHOSPHORUS, PLANTAGO, PULSATILLA, SABAL, TEREBINTHINA, UVA-URSI, ECHINACEA, HYDRASTIS, LAPPA, SOLIDAGO', 'BENZOCAINE AND DEXTROMETHORPHAN HBR', 'MEPERIDINE HYDROCHLORIDE'] prompt_tokens=1966 truncated=False
- There is no evidence that divalproex sodium extended-release tablets are useful in the acute treatment of migraine headaches. 1.4 Important Limitations Because of the risk to the fetus of decreased IQ, neurodevelopmental disorders, neural tube defects, and other major congenital malformations, which may occur very early in pregna

In [15]:
# ===== 13. summary =====
df = pd.DataFrame([{"qid": k, "type": k.split("-")[0], **{x: v.get(x) for x in
      ["drug_recall", "drug_recall_ing", "section_recall", "abstain_correct", "unsupported_rate", "prompt_tokens", "seconds"]}} for k, v in RESULTS.items()])
print(df.groupby("type").agg(n=("qid", "count"), drug_recall=("drug_recall", "mean"), drug_recall_ing=("drug_recall_ing", "mean"),
      section_recall=("section_recall", "mean"), abstain_correct=("abstain_correct", "mean"), unsupported_rate=("unsupported_rate", "mean"),
      max_prompt_tokens=("prompt_tokens", "max"), avg_seconds=("seconds", "mean")).round(2))
R = list(RESULTS.values())
print("\nstarts with Reasoning:", sum(r["raw_answer"].lstrip().startswith("Reasoning:") for r in R), "of", len(R))
print("truncated:", sum(bool(r.get("truncated")) for r in R), "| retried longer:", sum(bool(r.get("retried_longer")) for r in R),
      "| no answer parsed:", [k for k, r in RESULTS.items() if r.get("abstain_reason", "") and "no answer parsed" in r["abstain_reason"]])
print("gold drug not retrieved:", list(df[df.drug_recall_ing < 1].qid))
print("Now hand-score final_answer vs answer_key (correct / partially correct / wrong), same rule as V3.")

      n  drug_recall  drug_recall_ing  section_recall  abstain_correct  \
type                                                                     
T1    6         0.67             0.67            0.94             1.00   
T2    8         0.00             0.00            0.81             1.00   
T3    9         0.67             0.67            0.94             1.00   
T4    9         0.11             0.11            0.89             1.00   
T5    9         0.78             0.78            0.83             1.00   
T6    9         1.00             1.00             NaN             0.11   
T7    5         1.00             1.00            1.00             1.00   
T8    5         0.60             0.60            0.79             0.80   

      unsupported_rate  max_prompt_tokens  avg_seconds  
type                                                    
T1                0.00               1405         6.62  
T2                0.00               2057        21.32  
T3                0.00         